In [1]:
import yfinance as yf

tickers = ['AAPL', 'MSFT', 'GOOGL', 'TSLA', "ITX.MC"]

# Por hora
data = yf.download(tickers, period='7d', interval='1h')
print("\n=== PRECIOS POR HORA ===")
print(data['Close'])

# Por día
data = yf.download(tickers, period='1y', interval='1d')
print("\n=== PRECIOS POR DÍA ===")
print(data['Close'])

C:\Users\ivan\AppData\Local\Temp\ipykernel_7172\3745234169.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, period='7d', interval='1h')
[*********************100%***********************]  5 of 5 completed
C:\Users\ivan\AppData\Local\Temp\ipykernel_7172\3745234169.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, period='1y', interval='1d')



=== PRECIOS POR HORA ===
Ticker                           AAPL       GOOGL     ITX.MC        MSFT  \
Datetime                                                                   
2026-08-21 07:00:00+00:00         NaN         NaN  57.639999         NaN   
2026-08-21 08:00:00+00:00         NaN         NaN  57.880001         NaN   
2026-08-21 09:00:00+00:00         NaN         NaN  57.799999         NaN   
2026-08-21 10:00:00+00:00         NaN         NaN  57.779999         NaN   
2026-08-21 11:00:00+00:00         NaN         NaN  57.880001         NaN   
...                               ...         ...        ...         ...   
2026-08-31 15:30:00+00:00  314.670013  338.329987        NaN  510.600098   
2026-08-31 16:30:00+00:00  313.880005  338.369995        NaN  510.679901   
2026-08-31 17:30:00+00:00  315.684998  338.049988        NaN  510.920013   
2026-08-31 18:30:00+00:00  316.024994  338.600098        NaN  510.595001   
2026-08-31 19:30:00+00:00  317.230011  339.279999        NaN  

[*********************100%***********************]  5 of 5 completed


=== PRECIOS POR DÍA ===
Ticker            AAPL       GOOGL     ITX.MC        MSFT        TSLA
Date                                                                 
2025-09-01         NaN         NaN  40.750950         NaN         NaN
2025-09-02  228.874542  210.748795  39.852249  500.997925  329.359985
2025-09-03  237.592331  230.003845  40.799274  501.226074  334.089996
2025-09-04  238.897507  231.639175  41.234123  503.824738  338.529999
2025-09-05  238.807846  234.331528  40.654320  490.960571  350.839996
...                ...         ...        ...         ...         ...
2026-08-25  309.899994  346.959991  58.279999  491.709991  350.250000
2026-08-26  313.450012  342.000000  58.259998  496.369995  345.820007
2026-08-27  314.579987  340.649994  57.700001  505.059998  354.809998
2026-08-28  319.700012  346.589996  58.320000  513.530029  348.750000
2026-08-31  316.850006  339.350006  58.000000  507.290009  367.950012

[258 rows x 5 columns]


In [ ]:
import pandas as pd
def get_prices(self, tickers: list[str], period: str = "1y", interval: str = "1d") -> pd.DataFrame:
    data = yf.download(tickers, period=period, interval=interval)
    if data.empty:
        return pd.DataFrame(columns=["ticker", "date", "open", "high", "low", "close", "volume"])
    data = data.stack(level=1).rename_axis(["date", "ticker"]).reset_index()
    data = data.dropna(subset=["Open"])
    return data[["ticker", "date", "Open", "High", "Low", "Close", "Volume"]].rename(
        columns={"Open": "open", "High": "high", "Low": "low", "Close": "close", "Volume": "volume"}
    )
result = get_prices(None, tickers, period='7d', interval='1h')
print("\n=== PRECIOS POR HORA (DataFrame) ===")
display(result)


def add_variations(df: pd.DataFrame, var: str = "close") -> pd.DataFrame:
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    
    for ticker in df['ticker'].unique():
        mask = df['ticker'] == ticker
        ticker_data = df[mask].sort_values('date')
        
        df.loc[mask, 'var_1m'] = ticker_data[var].pct_change(periods=30) * 100
        df.loc[mask, 'var_3m'] = ticker_data[var].pct_change(periods=90) * 100
        df.loc[mask, 'var_6m'] = ticker_data[var].pct_change(periods=180) * 100
        df.loc[mask, 'var_9m'] = ticker_data[var].pct_change(periods=270) * 100
    
    return df[df.groupby('ticker').cumcount(ascending=False) == 0]

result = get_prices(None, tickers, period='1y', interval='1d')
result = add_variations(result)
display(result)

C:\Users\ivan\AppData\Local\Temp\ipykernel_7172\3493023496.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, period=period, interval=interval)
[*********************100%***********************]  5 of 5 completed


=== PRECIOS POR HORA (DataFrame) ===


Price,ticker,date,open,high,low,close,volume
2,ITX.MC,2026-08-21 07:00:00+00:00,57.779999,57.840000,57.439999,57.639999,0.0
7,ITX.MC,2026-08-21 08:00:00+00:00,57.639999,57.900002,57.560001,57.880001,23218.0
12,ITX.MC,2026-08-21 09:00:00+00:00,57.900002,57.900002,57.740002,57.799999,115649.0
17,ITX.MC,2026-08-21 10:00:00+00:00,57.779999,57.860001,57.740002,57.779999,13022.0
22,ITX.MC,2026-08-21 11:00:00+00:00,57.759998,57.880001,57.740002,57.880001,10341.0
...,...,...,...,...,...,...,...
554,TSLA,2026-08-31 18:30:00+00:00,367.420013,367.454987,365.250000,366.665009,4510217.0
555,AAPL,2026-08-31 19:30:00+00:00,316.029999,317.690002,315.709991,317.230011,5002995.0
556,GOOGL,2026-08-31 19:30:00+00:00,338.600006,341.420013,338.334991,339.279999,4566155.0
558,MSFT,2026-08-31 19:30:00+00:00,510.570007,510.595001,506.399994,507.320007,3800862.0


C:\Users\ivan\AppData\Local\Temp\ipykernel_7172\3493023496.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, period=period, interval=interval)
[*********************100%***********************]  5 of 5 completed


Price,ticker,date,open,high,low,close,volume,var_1m,var_3m,var_6m,var_9m
1285,AAPL,2026-08-31,319.559998,321.234985,312.799988,316.850006,40667429.0,-2.898654,16.197023,13.965302,NaN
1286,GOOGL,2026-08-31,343.697998,344.589996,337.156006,339.350006,33315062.0,-3.591007,0.068573,6.115278,NaN
1287,ITX.MC,2026-08-31,58.400002,58.520000,57.820000,58.000000,1564921.0,8.532937,12.243890,5.802357,NaN
1288,MSFT,2026-08-31,510.325012,512.190002,506.399994,507.290009,26637042.0,26.337959,17.653555,6.675929,NaN
1289,TSLA,2026-08-31,347.149994,368.920013,347.149994,367.950012,61157428.0,-0.438346,-5.047611,-18.495957,NaN
